In [33]:
!pip install gradio crewai litellm -q


In [34]:
from getpass import getpass
import subprocess

token = getpass("Paste new token and press Enter: ")

# Update the remote URL with the token embedded
repo_url = f"https://{token}@github.com/Highflyer1919/EduVerse.git"
subprocess.run(["git", "remote", "set-url", "origin", repo_url])

# Push
result = subprocess.run(["git", "push", "-u", "origin", "main"],
                        capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

Paste new token and press Enter: ··········
Branch 'main' set up to track remote branch 'main' from 'origin'.

To https://github.com/Highflyer1919/EduVerse.git
 * [new branch]      main -> main



In [35]:
import os, random, gradio as gr

os.environ["OPENAI_API_KEY"] = (
    "sk-proj-_WKaxhqS2FM4HWp8RKl1ba9p3-m28y84gCYUfrYBawxc3LFX4T8YfWgUIh"
    "QCRnQbtIwNSO23UpT3BlbkFJX_lpxGmvRYJFokfPN4zNbbbNbALkuV4OcFLnKxcIm2"
    "XAxNLP8SSpWvX0Niv70vWJUn7Am6mcgA"
)

# ─────────────────────────────────────────────────────────────
# QUESTION BANK
# ─────────────────────────────────────────────────────────────
QUESTION_BANK = [
    {"subtopic":"Limits",      "difficulty":"easy",
     "question":"What is lim(x\u21922) of (x\u00b2 \u2212 4)/(x \u2212 2)?",
     "answer":"4",
     "explanation":"Factor as (x\u22122)(x+2), cancel (x\u22122), sub x=2 \u2192 4."},
    {"subtopic":"Limits",      "difficulty":"easy",
     "question":"Evaluate lim(x\u21920) sin(x)/x.",
     "answer":"1",
     "explanation":"Standard trigonometric limit result equals 1."},
    {"subtopic":"Derivatives", "difficulty":"easy",
     "question":"Find the derivative of f(x) = 5x\u00b3.",
     "answer":"15x\u00b2",
     "explanation":"Power rule: d/dx[x\u207f] = n\u00b7x\u207f\u207b\u00b9 \u2192 15x\u00b2."},
    {"subtopic":"Derivatives", "difficulty":"easy",
     "question":"What is d/dx[sin(x)]?",
     "answer":"cos(x)",
     "explanation":"Standard derivative: d/dx[sin x] = cos x."},
    {"subtopic":"Integrals",   "difficulty":"easy",
     "question":"Evaluate \u222b 3x\u00b2 dx.",
     "answer":"x\u00b3 + C",
     "explanation":"\u222bx\u207fdx = x\u207f\u207a\u00b9/(n+1)+C \u2192 x\u00b3+C."},
    {"subtopic":"Limits",      "difficulty":"easy",
     "question":"What is lim(x\u2192\u221e) 1/x?",
     "answer":"0",
     "explanation":"As x grows without bound, 1/x approaches 0."},
    {"subtopic":"Derivatives", "difficulty":"medium",
     "question":"Differentiate f(x) = x\u00b2\u00b7sin(x) using the product rule.",
     "answer":"2x\u00b7sin(x) + x\u00b2\u00b7cos(x)",
     "explanation":"Product rule: u=x\u00b2, u'=2x; v=sin x, v'=cos x."},
    {"subtopic":"Limits",      "difficulty":"medium",
     "question":"Evaluate lim(x\u21920) (1 \u2212 cos x)/x\u00b2.",
     "answer":"1/2",
     "explanation":"Standard result: (1\u2212cos x)/x\u00b2 \u2192 1/2."},
    {"subtopic":"Integrals",   "difficulty":"medium",
     "question":"Compute \u222b x\u00b7e\u02e3 dx.",
     "answer":"xe\u02e3 \u2212 e\u02e3 + C",
     "explanation":"Integration by parts: u=x, dv=e\u02e3dx."},
    {"subtopic":"Derivatives", "difficulty":"medium",
     "question":"Find dy/dx if y = ln(x\u00b2 + 1).",
     "answer":"2x/(x\u00b2+1)",
     "explanation":"Chain rule: d/dx[ln u]=u'/u, u=x\u00b2+1, u'=2x."},
    {"subtopic":"Integrals",   "difficulty":"medium",
     "question":"Evaluate \u222b\u2080\u00b9 (2x + 1) dx.",
     "answer":"2",
     "explanation":"[x\u00b2+x]\u2080\u00b9 = 2 \u2212 0 = 2."},
    {"subtopic":"Derivatives", "difficulty":"medium",
     "question":"Find the derivative of f(x) = e\u02e3\u00b7cos(x).",
     "answer":"e\u02e3\u00b7cos(x) \u2212 e\u02e3\u00b7sin(x)",
     "explanation":"Product rule on e\u02e3 and cos x."},
    {"subtopic":"Integrals",   "difficulty":"hard",
     "question":"Evaluate \u222b sin\u00b2(x) dx.",
     "answer":"x/2 \u2212 sin(2x)/4 + C",
     "explanation":"Use identity sin\u00b2x=(1\u2212cos2x)/2, integrate term-by-term."},
    {"subtopic":"Derivatives", "difficulty":"hard",
     "question":"Differentiate f(x) = (sin x)^(cos x).",
     "answer":"(sin x)^(cos x)\u00b7[cos x\u00b7cot x \u2212 sin x\u00b7ln(sin x)]",
     "explanation":"Logarithmic differentiation: ln f = cos x\u00b7ln(sin x)."},
    {"subtopic":"Limits",      "difficulty":"hard",
     "question":"Find lim(x\u2192\u221e) x\u00b7sin(1/x).",
     "answer":"1",
     "explanation":"Let t=1/x \u2192 lim(t\u21920) sin(t)/t = 1."},
    {"subtopic":"Integrals",   "difficulty":"hard",
     "question":"Evaluate \u222b dx/(x\u00b2 \u2212 1).",
     "answer":"(1/2)\u00b7ln|(x\u22121)/(x+1)| + C",
     "explanation":"Partial fractions: 1/(x\u00b2\u22121)=(1/2)[1/(x\u22121)\u22121/(x+1)]."},
    {"subtopic":"Derivatives", "difficulty":"hard",
     "question":"Find all critical points of f(x) = x\u00b3 \u2212 6x\u00b2 + 9x + 1.",
     "answer":"x=1 (local max), x=3 (local min)",
     "explanation":"f'=3x\u00b2\u221212x+9=3(x\u22121)(x\u22123)=0 \u2192 x=1,3."},
    {"subtopic":"Limits",      "difficulty":"hard",
     "question":"Evaluate lim(x\u21920\u207a) x\u00b7ln(x).",
     "answer":"0",
     "explanation":"Rewrite as ln(x)/(1/x), L'H\u00f4pital \u2192 \u2212x \u2192 0."},
]

# ─────────────────────────────────────────────────────────────
# LOGIC HELPERS
# ─────────────────────────────────────────────────────────────
def classify_student(accuracy):
    if accuracy < 0.4:   return "Beginner"
    elif accuracy < 0.7: return "Intermediate"
    else:                return "Advanced"

def reflection_text(profile, weak):
    acc = profile["accuracy"]
    lines = []
    if acc < 0.3:
        lines += ["Diagnosis:  Struggling with fundamental concepts.",
                  "Suggestion: Review core definitions before harder problems."]
    elif acc < 0.6:
        lines += ["Diagnosis:  Understands some concepts but makes frequent mistakes.",
                  "Suggestion: Focus on medium-difficulty problem practice."]
    else:
        lines += ["Diagnosis:  Strong understanding -- needs refinement.",
                  "Suggestion: Try advanced or multi-step problems."]
    if weak:
        lines.append("\nLearning Gaps Detected In:")
        for topic, _ in weak[:2]:
            lines.append("  -> " + topic)
    return "\n".join(lines)

def study_plan_text(weak):
    if not weak:
        return "* Keep up the great work! Try harder difficulty next time."
    lines = []
    for topic, _ in weak[:3]:
        t = topic.lower()
        if   t == "integrals":    lines.append("* Practice substitution and basic integration rules.")
        elif t == "limits":       lines.append("* Review limit laws and practice indeterminate forms.")
        elif t == "derivatives":  lines.append("* Practice chain rule and product rule.")
        else:                     lines.append("* Practice more problems in " + topic + ".")
    return "\n".join(lines)

def run_crew(profile):
    try:
        from crewai import Agent, Task, Crew
        ctx = ("Accuracy: " + str(round(profile["accuracy"] * 100)) + "%, "
               + "Level: " + profile["level"] + ", "
               + "Weak topics: " + str(profile["weak_topics"]) + ", "
               + "Total errors: " + str(profile["total_errors"]))
        agents = [
            Agent(role="Diagnostic Tutor",
                  goal="Evaluate the student calculus knowledge",
                  backstory="Experienced tutor.", verbose=False),
            Agent(role="Student Modeling Specialist",
                  goal="Estimate skill level from diagnostic results",
                  backstory="AI learning researcher.", verbose=False),
            Agent(role="Weak Topic Analyzer",
                  goal="Identify weak calculus topics",
                  backstory="Education gap analyst.", verbose=False),
            Agent(role="Learning Coach",
                  goal="Create a personalised learning plan",
                  backstory="Student improvement coach.", verbose=False),
        ]
        tasks = [
            Task(description="Evaluate this student: " + ctx + ". Summarise performance.",
                 expected_output="Performance summary with strengths and weaknesses.",
                 agent=agents[0]),
            Task(description="Build a skill model for: " + ctx + ".",
                 expected_output="Proficiency per topic on 0-1 scale.", agent=agents[1]),
            Task(description="Rank weak topics for: " + ctx + ".",
                 expected_output="Ranked weak topics with justification.", agent=agents[2]),
            Task(description="Write a 3-step study plan for: " + ctx + ".",
                 expected_output="Concise 3-step personalised study plan.", agent=agents[3]),
        ]
        result = Crew(agents=agents, tasks=tasks, verbose=False).kickoff()
        return str(result)
    except ImportError:
        return "CrewAI not installed. Run: !pip install crewai litellm -q"
    except Exception as exc:
        return "CrewAI error: " + str(exc)

# ─────────────────────────────────────────────────────────────
# HTML BUILDERS  (plain string concatenation — no f-strings)
# ─────────────────────────────────────────────────────────────
def build_progress(done, total):
    pct = int(done / total * 100) if total else 0
    return (
        '<div style="margin-bottom:14px;">'
        '<div style="color:#555;font-size:0.72rem;letter-spacing:2px;margin-bottom:6px;'
        'font-family:Courier New,monospace;">QUESTION ' + str(done) + ' OF ' + str(total) + '</div>'
        '<div style="background:#1e1e1e;border-radius:99px;height:5px;width:100%;">'
        '<div style="background:linear-gradient(90deg,#f97316,#eab308);height:5px;'
        'border-radius:99px;width:' + str(pct) + '%;"></div>'
        '</div></div>'
    )

def build_question(q, num, total):
    dc = {"easy":"#22c55e","medium":"#eab308","hard":"#f97316"}.get(q["difficulty"], "#aaa")
    return (
        '<div style="background:#161616;border:1px solid #2a2a2a;border-left:4px solid #f97316;'
        'border-radius:6px;padding:20px 24px;">'
        '<div style="display:flex;gap:12px;align-items:center;margin-bottom:12px;'
        'font-family:Courier New,monospace;">'
        '<span style="color:#555;font-size:0.72rem;letter-spacing:2px;">Q '
        + str(num) + ' / ' + str(total) + '</span>'
        '<span style="background:#1e1e1e;color:#aaa;border-radius:3px;padding:2px 10px;'
        'font-size:0.72rem;">' + q["subtopic"] + '</span>'
        '<span style="color:' + dc + ';font-size:0.72rem;letter-spacing:2px;font-weight:bold;">'
        + q["difficulty"].upper() + '</span>'
        '</div>'
        '<div style="font-size:1.1rem;color:#fff;line-height:1.7;'
        'font-family:Courier New,monospace;">' + q["question"] + '</div>'
        '</div>'
    )

def build_feedback(is_correct, user_answer, correct_answer, explanation):
    if is_correct:
        return (
            '<div style="background:#052e16;border:1px solid #166534;border-radius:6px;'
            'padding:14px 18px;color:#4ade80;font-family:Courier New,monospace;margin-top:8px;">'
            '<strong>Correct!</strong><br>'
            '<span style="color:#aaa;font-size:0.84rem;">Your answer: ' + user_answer + '</span>'
            '</div>'
        )
    return (
        '<div style="background:#1c0a00;border:1px solid #7c2d12;border-radius:6px;'
        'padding:14px 18px;color:#fb923c;font-family:Courier New,monospace;margin-top:8px;">'
        '<strong>Incorrect.</strong><br>'
        '<span style="color:#aaa;font-size:0.84rem;">Your answer: ' + user_answer + '</span><br>'
        '<span style="color:#aaa;font-size:0.84rem;">Correct answer: <strong>'
        + correct_answer + '</strong></span><br>'
        '<span style="color:#aaa;font-size:0.84rem;">' + explanation + '</span>'
        '</div>'
    )

def build_results(session):
    total  = len(session["questions"])
    score  = session["score"]
    errors = session["errors"]
    acc    = score / total if total else 0
    lvl    = classify_student(acc)
    weak   = sorted(errors.items(), key=lambda x: x[1], reverse=True)
    lc     = {"Beginner":"#f97316","Intermediate":"#eab308","Advanced":"#22c55e"}.get(lvl,"#aaa")
    pre    = ("background:#0a0a0a;border:1px solid #222;border-radius:6px;padding:14px;"
              "font-size:0.8rem;color:#ccc;white-space:pre-wrap;line-height:1.8;"
              "font-family:Courier New,monospace;")

    def sec(t):
        return ('<div style="font-size:0.72rem;letter-spacing:3px;color:#f97316;'
                'text-transform:uppercase;border-left:3px solid #f97316;padding-left:10px;'
                'margin:20px 0 10px;font-family:Courier New,monospace;">' + t + '</div>')

    score_card = (
        '<div style="display:flex;align-items:center;gap:24px;background:#161616;'
        'border:1px solid #2a2a2a;border-radius:8px;padding:22px 30px;margin-bottom:20px;'
        'font-family:Courier New,monospace;">'
        '<div style="font-size:3rem;color:#f97316;font-weight:bold;line-height:1;">'
        + str(score) + '/' + str(total) + '</div>'
        '<div style="font-size:1.4rem;color:#555;">' + str(round(acc * 100)) + '%</div>'
        '<div style="border:2px solid ' + lc + ';color:' + lc + ';border-radius:20px;'
        'padding:5px 18px;font-size:0.78rem;letter-spacing:2px;">' + lvl.upper() + '</div>'
        '</div>'
    )

    if weak:
        rows = "".join(
            '<tr>'
            '<td style="padding:8px 12px;border-bottom:1px solid #1a1a1a;'
            'font-family:Courier New,monospace;">' + t + '</td>'
            '<td style="padding:8px 12px;border-bottom:1px solid #1a1a1a;'
            'color:#f97316;font-family:Courier New,monospace;">'
            + str(e) + " error" + ("s" if e > 1 else "") + '</td></tr>'
            for t, e in weak
        )
        wt = (
            '<table style="width:100%;border-collapse:collapse;font-size:0.86rem;">'
            '<thead><tr>'
            '<th style="text-align:left;color:#555;padding:6px 12px;border-bottom:1px solid #222;'
            'font-size:0.7rem;letter-spacing:2px;font-family:Courier New,monospace;">TOPIC</th>'
            '<th style="text-align:left;color:#555;padding:6px 12px;border-bottom:1px solid #222;'
            'font-size:0.7rem;letter-spacing:2px;font-family:Courier New,monospace;">ERRORS</th>'
            '</tr></thead><tbody>' + rows + '</tbody></table>'
        )
    else:
        wt = ('<p style="color:#22c55e;font-family:Courier New,monospace;margin:0;">'
              'No weak topics -- excellent work!</p>')

    profile = {"accuracy": acc, "level": lvl,
               "total_errors": sum(errors.values()), "weak_topics": errors}
    return (
        score_card
        + sec("Weak Topic Breakdown") + wt
        + sec("Reflection Agent")
        + '<pre style="' + pre + '">' + reflection_text(profile, weak) + '</pre>'
        + sec("Recommended Study Plan")
        + '<pre style="' + pre + '">' + study_plan_text(weak) + '</pre>'
    )

# ─────────────────────────────────────────────────────────────
# SCIENTIFIC CALCULATOR  (r-string — no Python escape issues)
#
# HOW IT REACHES THE GRADIO TEXTAREA:
#   Gradio 4.44.1 places elem_id on the outermost Block <div>.
#   Inside that div Gradio renders:
#     <label class="container">
#       <span>label</span>
#       <textarea data-testid="textbox">   ← THIS is our target
#     </label>
#   With lines=2 / max_lines=2 Gradio always renders a <textarea>.
#   Selector: #answer-wrap textarea[data-testid="textbox"]
#   After setting .value we dispatch a native "input" event so
#   Svelte's bind:value picks up the change.
# ─────────────────────────────────────────────────────────────
CALC_JS = r"""
(function () {
  var expr = "";
  var justUsed = false;

  /* ── display helpers ── */
  function getDisplay() {
    var d = document.getElementById("calc-display");
    return d ? d.textContent : "0";
  }
  function setDisplay(val) {
    var d = document.getElementById("calc-display");
    if (d) d.textContent = (val === "" || val === null || val === undefined) ? "0" : String(val);
  }

  /* ── find Gradio textarea ──
     #answer-wrap  →  Block wrapper (where elem_id lands)
     textarea[data-testid="textbox"]  →  the actual input element
  */
  function getTA() {
    var ta = document.querySelector('#answer-wrap textarea[data-testid="textbox"]');
    if (!ta) ta = document.querySelector('#quiz-col textarea[data-testid="textbox"]');
    return ta;
  }

  /* ── calculator operations ── */
  function press(token) {
    if (justUsed) { expr = ""; justUsed = false; }
    expr += token;
    setDisplay(expr);
  }
  function back() {
    expr = expr.slice(0, -1);
    setDisplay(expr || "");
  }
  function clear() {
    expr = "";
    setDisplay("0");
  }

  /* evaluate: only numeric expressions; algebraic ones go straight to useAnswer */
  function evaluate() {
    if (!expr) return;
    if (/[a-wyzA-WYZ]/.test(expr)) { useAnswer(); return; }
    try {
      var js = expr
        .replace(/\u00d7/g, "*")
        .replace(/\u00f7/g, "/")
        .replace(/\u03c0/g, String(Math.PI))
        .replace(/e/g, String(Math.E))
        .replace(/\u221a(\d+\.?\d*)/g, "Math.sqrt($1)")
        .replace(/\^/g, "**")
        .replace(/sin\(/g, "Math.sin(")
        .replace(/cos\(/g, "Math.cos(")
        .replace(/tan\(/g, "Math.tan(")
        .replace(/log\(/g, "Math.log10(")
        .replace(/ln\(/g, "Math.log(");
      var result = Function('"use strict"; return (' + js + ')')();
      if (!isFinite(result)) { setDisplay("Error"); expr = ""; return; }
      expr = String(parseFloat(result.toPrecision(10)));
      setDisplay(expr);
    } catch (ex) {
      setDisplay("Error");
      expr = "";
    }
  }

  /* useAnswer: copy expression into Gradio textarea and trigger its update */
  function useAnswer() {
    var ta = getTA();
    if (!ta) {
      alert("Answer box not found.\nPlease click once inside the answer box above, then press USE ANSWER.");
      return;
    }
    var val = (expr === "" || expr === undefined) ? getDisplay() : expr;
    ta.value = val;
    ta.focus();
    /* Svelte bind:value listens to the native 'input' event */
    ta.dispatchEvent(new Event("input", { bubbles: true }));
    justUsed = true;
    /* Visual confirmation */
    var btn = document.getElementById("calc-use-btn");
    if (btn) {
      btn.style.background = "#22c55e";
      btn.textContent = "\u2713  SENT TO ANSWER BOX";
      setTimeout(function () {
        btn.style.background = "#f97316";
        btn.textContent = "USE ANSWER  \u2191";
      }, 1400);
    }
  }

  /* ── button definitions: [label, type, token] ── */
  var BTNS = [
    ["sin(","sci","sin("],  ["cos(","sci","cos("],  ["tan(","sci","tan("],
    ["log(","sci","log("],  ["ln(", "sci","ln("],
    ["\u00b2","sci","\u00b2"], ["\u00b3","sci","\u00b3"], ["\u221a","sci","\u221a"],
    ["\u03c0","sci","\u03c0"], ["e","sci","e"],
    ["(","bracket","("],    [")","bracket",")"],    ["^","op","^"],
    ["\u00b1","op","-"],    ["%","op","%"],
    ["7","num","7"], ["8","num","8"], ["9","num","9"], ["\u00f7","op","\u00f7"], ["C","clear","C"],
    ["4","num","4"], ["5","num","5"], ["6","num","6"], ["\u00d7","op","\u00d7"], ["DEL","del","DEL"],
    ["1","num","1"], ["2","num","2"], ["3","num","3"], ["-","op","-"],            ["=","eq","="],
    ["0","num","0"], [".","num","."], ["x","var","x"],["+","op","+"],[" ","blank",""]
  ];

  /* ── colour map ── */
  var COL = {
    num:     "background:#1a1a1a;color:#e8e8e8;",
    op:      "background:#2a1500;color:#f97316;",
    sci:     "background:#001525;color:#38bdf8;",
    bracket: "background:#12102a;color:#a78bfa;",
    eq:      "background:#052e16;color:#4ade80;",
    clear:   "background:#2a0500;color:#f97316;",
    del:     "background:#1a0505;color:#fb923c;",
    var:     "background:#1a1500;color:#eab308;",
    blank:   "background:transparent;border:none;cursor:default;"
  };

  /* ── build button grid ── */
  var grid = document.getElementById("calc-grid");
  if (grid) {
    BTNS.forEach(function (btnDef) {
      var label = btnDef[0], type = btnDef[1], token = btnDef[2];
      var el = document.createElement("button");
      el.type = "button";
      el.textContent = label;
      el.style.cssText = (
        "border:1px solid #2a2a2a;border-radius:4px;padding:9px 2px;" +
        "font-size:0.9rem;cursor:pointer;font-family:Courier New,monospace;" +
        "transition:filter 0.1s;width:100%;" +
        (COL[type] || COL.num)
      );
      if (type !== "blank") {
        el.addEventListener("mouseenter", function () { this.style.filter = "brightness(1.5)"; });
        el.addEventListener("mouseleave", function () { this.style.filter = ""; });
      }
      if      (type === "clear") { el.addEventListener("click", function () { clear();    }); }
      else if (type === "del")   { el.addEventListener("click", function () { back();     }); }
      else if (type === "eq")    { el.addEventListener("click", function () { evaluate(); }); }
      else if (type === "blank") { el.style.pointerEvents = "none"; }
      else {
        (function (t) {
          el.addEventListener("click", function () { press(t); });
        })(token);
      }
      grid.appendChild(el);
    });
  }

  /* ── wire USE ANSWER button ── */
  var useBtn = document.getElementById("calc-use-btn");
  if (useBtn) {
    useBtn.addEventListener("click", function () { useAnswer(); });
    useBtn.addEventListener("mouseenter", function () { this.style.filter = "brightness(1.15)"; });
    useBtn.addEventListener("mouseleave", function () { this.style.filter = ""; });
  }
})();
"""

CALC_HTML = (
    '<div id="sci-calc" style="background:#0a0a0a;border:1px solid #2a2a2a;'
    'border-radius:8px;padding:16px;margin-top:8px;">'
    # header
    '<div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:10px;">'
    '<span style="color:#f97316;font-size:0.72rem;letter-spacing:3px;'
    'font-family:Courier New,monospace;text-transform:uppercase;">Scientific Calculator</span>'
    '<span style="color:#555;font-size:0.68rem;font-family:Courier New,monospace;">'
    'Build expression \u2192 press USE ANSWER</span>'
    '</div>'
    # display
    '<div id="calc-display" style="background:#000;border:1px solid #333;border-radius:4px;'
    'padding:12px 16px;font-size:1.25rem;color:#f97316;text-align:right;min-height:50px;'
    'margin-bottom:10px;font-family:Courier New,monospace;letter-spacing:1px;'
    'word-break:break-all;">0</div>'
    # button grid (populated by JS)
    '<div id="calc-grid" style="display:grid;grid-template-columns:repeat(5,1fr);gap:5px;"></div>'
    # use-answer button
    '<button id="calc-use-btn" type="button" '
    'style="width:100%;margin-top:8px;padding:10px;background:#f97316;color:#000;'
    'border:none;border-radius:4px;font-size:0.88rem;font-weight:bold;cursor:pointer;'
    'font-family:Courier New,monospace;letter-spacing:2px;transition:filter 0.15s;">'
    'USE ANSWER \u2191'
    '</button>'
    '</div>'
    '<script>' + CALC_JS + '</script>'
)

# ─────────────────────────────────────────────────────────────
# SESSION
# ─────────────────────────────────────────────────────────────
def new_session():
    return {"questions": [], "idx": 0, "score": 0, "errors": {}, "phase": "setup"}

# ─────────────────────────────────────────────────────────────
# CSS
# ─────────────────────────────────────────────────────────────
CSS = """
* { box-sizing: border-box; }
body, .gradio-container { background: #0d0d0d !important; color: #e8e8e8 !important; }
.gr-button {
    font-family: 'Courier New', monospace !important;
    font-weight: bold !important;
    letter-spacing: 1px !important;
    border-radius: 4px !important;
    transition: all 0.18s !important;
}
.btn-primary         { background: #f97316 !important; color: #000 !important; border: none !important; }
.btn-primary:hover   { background: #ea6a0a !important; }
.btn-secondary       { background: transparent !important; color: #f97316 !important; border: 1px solid #f97316 !important; }
.btn-secondary:hover { background: #1a0e00 !important; }
.panel               { background: #111 !important; border: 1px solid #1e1e1e !important; border-radius: 8px !important; padding: 24px !important; }
#answer-wrap textarea {
    background: #111 !important;
    border: 1px solid #333 !important;
    color: #e8e8e8 !important;
    border-radius: 4px !important;
    font-family: 'Courier New', monospace !important;
    font-size: 1.05rem !important;
    padding: 10px 14px !important;
    caret-color: #f97316 !important;
    resize: none !important;
}
#answer-wrap textarea:focus {
    border-color: #f97316 !important;
    box-shadow: 0 0 0 2px rgba(249,115,22,0.25) !important;
    outline: none !important;
}
.gr-textbox textarea, .gr-textbox input {
    background: #111 !important; border: 1px solid #2a2a2a !important;
    color: #e8e8e8 !important; border-radius: 4px !important;
    font-family: 'Courier New', monospace !important;
}
label span, .gr-form label span {
    color: #555 !important; font-size: 0.72rem !important;
    letter-spacing: 2px !important; font-family: 'Courier New', monospace !important;
}
.gr-slider input { accent-color: #f97316; }
.gr-radio label  { color: #aaa !important; font-family: 'Courier New', monospace !important; }
"""

# ─────────────────────────────────────────────────────────────
# GRADIO APP
# ─────────────────────────────────────────────────────────────
with gr.Blocks(css=CSS, title="AI Calculus Tutor") as demo:

    state = gr.State(new_session())

    gr.HTML(
        '<div style="text-align:center;padding:28px 0 18px;border-bottom:1px solid #1e1e1e;margin-bottom:24px;">'
        '<h1 style="font-family:Courier New,monospace;font-size:1.9rem;letter-spacing:5px;'
        'color:#f97316;margin:0;text-transform:uppercase;">AI Calculus Tutor</h1>'
        '<p style="color:#444;font-size:0.78rem;letter-spacing:3px;margin:6px 0 0;'
        'font-family:Courier New,monospace;">DIAGNOSTIC &middot; ADAPTIVE &middot; CREW-AI POWERED</p>'
        '</div>'
    )

    # ── SETUP ────────────────────────────────────────────────
    with gr.Column(visible=True, elem_classes=["panel"]) as setup_panel:
        gr.HTML(
            '<div style="font-size:0.72rem;letter-spacing:3px;color:#f97316;text-transform:uppercase;'
            'border-left:3px solid #f97316;padding-left:10px;margin-bottom:20px;'
            'font-family:Courier New,monospace;">Configure Session</div>'
        )
        with gr.Row():
            num_q_slider = gr.Slider(minimum=3, maximum=15, value=5, step=1,
                                     label="NUMBER OF QUESTIONS")
            diff_radio   = gr.Radio(["Adaptive (Auto)", "easy", "medium", "hard"],
                                    value="Adaptive (Auto)", label="DIFFICULTY MODE")
        start_btn = gr.Button("BEGIN DIAGNOSTIC TEST", elem_classes=["btn-primary"])

    # ── QUIZ ─────────────────────────────────────────────────
    with gr.Column(visible=False, elem_classes=["panel"], elem_id="quiz-col") as quiz_panel:
        progress_out = gr.HTML("")
        question_out = gr.HTML("")
        gr.HTML(
            '<div style="font-size:0.72rem;letter-spacing:2px;color:#555;'
            'font-family:Courier New,monospace;margin-bottom:4px;text-transform:uppercase;">'
            'Your Answer</div>'
        )
        # lines=2, max_lines=2 → Gradio renders <textarea data-testid="textbox">
        # elem_id="answer-wrap" → the Block wrapper div gets id="answer-wrap"
        # JS selector: #answer-wrap textarea[data-testid="textbox"]
        answer_in = gr.Textbox(
            show_label=False,
            placeholder="Use the calculator below, then press USE ANSWER \u2191",
            lines=2,
            max_lines=2,
            elem_id="answer-wrap",
        )
        gr.HTML(CALC_HTML)
        submit_btn   = gr.Button("Submit Answer", elem_classes=["btn-primary"])
        feedback_out = gr.HTML("")

    # ── RESULTS ──────────────────────────────────────────────
    with gr.Column(visible=False, elem_classes=["panel"]) as results_panel:
        gr.HTML(
            '<div style="font-size:0.72rem;letter-spacing:3px;color:#f97316;text-transform:uppercase;'
            'border-left:3px solid #f97316;padding-left:10px;margin-bottom:20px;'
            'font-family:Courier New,monospace;">Session Results</div>'
        )
        results_out = gr.HTML("")
        gr.HTML(
            '<div style="font-size:0.72rem;letter-spacing:3px;color:#f97316;text-transform:uppercase;'
            'border-left:3px solid #f97316;padding-left:10px;margin:24px 0 12px;'
            'font-family:Courier New,monospace;">CrewAI Multi-Agent Analysis</div>'
        )
        crew_out = gr.Textbox(
            label="CREW OUTPUT", lines=12, interactive=False,
            value="Click 'Run CrewAI Agents' for a full multi-agent analysis."
        )
        with gr.Row():
            crew_btn    = gr.Button("Run CrewAI Agents", elem_classes=["btn-primary"])
            restart_btn = gr.Button("New Session",       elem_classes=["btn-secondary"])

    # ── HANDLERS ─────────────────────────────────────────────
    def on_start(num_q, diff, _session):
        session = new_session()
        pool = QUESTION_BANK if diff == "Adaptive (Auto)" else \
               [q for q in QUESTION_BANK if q["difficulty"] == diff]
        if not pool:
            pool = QUESTION_BANK
        n = min(int(num_q), len(pool))
        session["questions"] = random.sample(pool, n)
        session["phase"] = "quiz"
        total = len(session["questions"])
        q = session["questions"][0]
        return (
            session,
            gr.update(visible=False),                           # setup_panel
            gr.update(visible=True),                            # quiz_panel
            gr.update(visible=False),                           # results_panel
            build_progress(0, total),                           # progress_out
            build_question(q, 1, total),                        # question_out
            gr.update(value="", interactive=True),              # answer_in
            gr.update(value="Submit Answer", interactive=True), # submit_btn
            gr.update(value=""),                                 # feedback_out
            gr.update(value=""),                                 # results_out
        )

    start_btn.click(
        fn=on_start,
        inputs=[num_q_slider, diff_radio, state],
        outputs=[state, setup_panel, quiz_panel, results_panel,
                 progress_out, question_out, answer_in, submit_btn,
                 feedback_out, results_out],
    )

    def on_submit(answer, session):
        if session["phase"] != "quiz":
            return (session,
                    gr.update(), gr.update(), gr.update(),
                    gr.update(), gr.update(), gr.update(),
                    gr.update(), gr.update(), gr.update())
        idx   = session["idx"]
        qs    = session["questions"]
        total = len(qs)
        q     = qs[idx]
        ok    = answer.strip().lower() in q["answer"].lower()
        if ok:
            session["score"] += 1
        else:
            session["errors"][q["subtopic"]] = session["errors"].get(q["subtopic"], 0) + 1
        fb = build_feedback(ok, answer, q["answer"], q["explanation"])
        session["idx"] += 1
        nxt = session["idx"]
        if nxt < total:
            nq = qs[nxt]
            return (
                session,
                gr.update(), gr.update(), gr.update(),          # panels unchanged
                build_progress(nxt, total),                     # progress_out
                build_question(nq, nxt + 1, total),             # question_out
                gr.update(value=""),                             # answer_in cleared
                gr.update(value="Submit Answer", interactive=True),
                gr.update(value=fb),                            # feedback_out
                gr.update(value=""),                             # results_out
            )
        else:
            session["phase"] = "done"
            return (
                session,
                gr.update(visible=False),                       # hide setup
                gr.update(visible=False),                       # hide quiz
                gr.update(visible=True),                        # show results
                gr.update(value=""),
                gr.update(value=""),
                gr.update(value=""),
                gr.update(interactive=False),
                gr.update(value=""),
                gr.update(value=build_results(session)),        # results_out
            )

    submit_btn.click(
        fn=on_submit,
        inputs=[answer_in, state],
        outputs=[state, setup_panel, quiz_panel, results_panel,
                 progress_out, question_out, answer_in, submit_btn,
                 feedback_out, results_out],
    )

    def on_crew(session):
        total  = len(session["questions"])
        score  = session["score"]
        errors = session["errors"]
        acc    = score / total if total else 0
        return run_crew({"accuracy": acc, "level": classify_student(acc),
                         "total_errors": sum(errors.values()), "weak_topics": errors})

    crew_btn.click(fn=on_crew, inputs=[state], outputs=[crew_out])

    def on_restart():
        return (
            new_session(),
            gr.update(visible=True),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(value=""),
            gr.update(value="Click 'Run CrewAI Agents' for a full multi-agent analysis."),
        )

    restart_btn.click(
        fn=on_restart,
        inputs=[],
        outputs=[state, setup_panel, quiz_panel, results_panel, results_out, crew_out],
    )

demo.launch(share=True, show_error=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://9e7d0e1a77abe46b61.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
